# Argo Excel Add-in Tutorial

This notebook demonstrates the **Argo Excel Add-in** for Monte Carlo simulations in Microsoft Excel (Office 365).

The Argo Excel add-in provides:
1. **Interactive Task Pane UI** - Full-featured simulation interface
2. **14 Custom Excel Functions** - Use `ARGO.NORMAL()`, `ARGO.UNIFORM()`, etc. in formulas
3. **Results Dashboard** - Histogram, CDF chart, and statistics
4. **Distribution Builder** - Visual parameter configuration

## Platform Support

The Argo add-in works on:
- **Excel for Windows** (Office 365)
- **Excel for Mac** (Office 365)
- **Excel on the Web** (Excel Online)

## Installation

### Development Install (Sideloading)

For development and testing:

```bash
# 1. Build the add-in
cd packages/argo-excel
npm run build

# 2. Start local dev server
npm run dev  # Runs on https://localhost:3000

# 3. Sideload manifest.xml in Excel
# - Windows/Mac: Insert > My Add-ins > Upload My Add-in
# - Web: Share > Settings > Manage Add-ins > Upload My Add-in
```

### Production Install (AppSource)

**Note:** Argo v5.0 will be published to Microsoft AppSource after Sprint 12 completion.

Once published:
1. Open Excel
2. Go to **Insert** > **Get Add-ins**
3. Search for **"Argo Monte Carlo Simulation"**
4. Click **Add**

## Part 1: Add-in Architecture Overview

The Argo Excel add-in is built with modern web technologies:

### Technology Stack

- **Office.js** - Excel integration API
- **React 18** - UI framework
- **Fluent UI** - Microsoft's design system
- **Recharts** - Data visualization (histogram, CDF)
- **TypeScript** - Type-safe development
- **Vite** - Fast build system
- **@argo/core** - Simulation engine (14 distributions)

### Package Structure

```
packages/argo-excel/
├── manifest.xml              # Office add-in manifest
├── src/
│   ├── taskpane/
│   │   ├── App.tsx          # Main application
│   │   ├── components/
│   │   │   ├── DistributionSelector.tsx
│   │   │   ├── SimulationControls.tsx
│   │   │   └── ResultsDashboard.tsx
│   │   └── index.tsx        # Task pane entry point
│   └── functions/
│       ├── functions.ts     # Custom function implementations
│       └── functions.json   # Custom function metadata
├── dist/                     # Built add-in (generated)
└── assets/                   # Icons and graphics
```

## Part 2: Office.js Integration

The add-in uses Office.js to interact with Excel. Here's how it works:

In [ ]:
// Example: Reading cell values from Excel
//
// NOTE: This code runs in the Excel add-in context, not Node.js
// It demonstrates the Office.js API pattern

const exampleReadCells = `
await Excel.run(async (context) => {
  // Get the active worksheet
  const sheet = context.workbook.worksheets.getActiveWorksheet();
  
  // Get a range (e.g., A1:B10)
  const range = sheet.getRange('A1:B10');
  
  // Load the values
  range.load('values');
  
  // Execute the queued commands
  await context.sync();
  
  // Access the values
  const values = range.values;
  console.log('Cell values:', values);
  
  return values;
});
`;

console.log('Office.js Read Pattern:');
console.log(exampleReadCells);

In [ ]:
// Example: Writing simulation results to Excel

const exampleWriteResults = `
await Excel.run(async (context) => {
  const sheet = context.workbook.worksheets.getActiveWorksheet();
  
  // Write headers
  const headers = sheet.getRange('A1:B1');
  headers.values = [['Statistic', 'Value']];
  headers.format.font.bold = true;
  headers.format.fill.color = '#01807E';  // Booz Allen teal
  headers.format.font.color = 'white';
  
  // Write simulation results
  const results = sheet.getRange('A2:B7');
  results.values = [
    ['Mean', 100.25],
    ['Median', 99.87],
    ['Std Dev', 15.03],
    ['Min', 45.23],
    ['Max', 155.67],
    ['Iterations', 10000]
  ];
  
  // Format numbers
  const valueRange = sheet.getRange('B2:B7');
  valueRange.numberFormat = [['0.00']];
  
  // Auto-fit columns
  headers.format.autofitColumns();
  
  await context.sync();
});
`;

console.log('Office.js Write Pattern:');
console.log(exampleWriteResults);

## Part 3: Custom Excel Functions

Argo provides 14 custom functions that can be used directly in Excel formulas.

### Function Naming Convention

All custom functions use the `ARGO.` namespace:
- `ARGO.NORMAL(mean, stddev)`
- `ARGO.UNIFORM(min, max)`
- `ARGO.TRIANGULAR(min, mode, max)`
- etc.

### Volatility

All Argo functions are **volatile** (recalculate on every sheet change), which is appropriate for Monte Carlo simulations.

### Available Custom Functions

In [ ]:
const customFunctions = [
  {
    name: 'ARGO.NORMAL',
    description: 'Returns a random sample from a Normal distribution',
    parameters: ['mean', 'stddev'],
    example: '=ARGO.NORMAL(100, 15)'
  },
  {
    name: 'ARGO.LOGNORMAL',
    description: 'Returns a random sample from a LogNormal distribution',
    parameters: ['mu', 'sigma'],
    example: '=ARGO.LOGNORMAL(4, 0.5)'
  },
  {
    name: 'ARGO.UNIFORM',
    description: 'Returns a random sample from a Uniform distribution',
    parameters: ['min', 'max'],
    example: '=ARGO.UNIFORM(0, 100)'
  },
  {
    name: 'ARGO.TRIANGULAR',
    description: 'Returns a random sample from a Triangular distribution',
    parameters: ['min', 'mode', 'max'],
    example: '=ARGO.TRIANGULAR(50, 100, 200)'
  },
  {
    name: 'ARGO.PERT',
    description: 'Returns a random sample from a PERT distribution',
    parameters: ['min', 'mode', 'max'],
    example: '=ARGO.PERT(50, 100, 200)'
  },
  {
    name: 'ARGO.EXPONENTIAL',
    description: 'Returns a random sample from an Exponential distribution',
    parameters: ['lambda'],
    example: '=ARGO.EXPONENTIAL(0.5)'
  },
  {
    name: 'ARGO.GAMMA',
    description: 'Returns a random sample from a Gamma distribution',
    parameters: ['shape', 'scale'],
    example: '=ARGO.GAMMA(2, 2)'
  },
  {
    name: 'ARGO.BETA',
    description: 'Returns a random sample from a Beta distribution',
    parameters: ['alpha', 'beta'],
    example: '=ARGO.BETA(2, 5)'
  },
  {
    name: 'ARGO.WEIBULL',
    description: 'Returns a random sample from a Weibull distribution',
    parameters: ['shape', 'scale'],
    example: '=ARGO.WEIBULL(2, 100)'
  },
  {
    name: 'ARGO.PARETO',
    description: 'Returns a random sample from a Pareto distribution',
    parameters: ['shape', 'scale'],
    example: '=ARGO.PARETO(2, 1)'
  },
  {
    name: 'ARGO.BINOMIAL',
    description: 'Returns a random sample from a Binomial distribution',
    parameters: ['n', 'p'],
    example: '=ARGO.BINOMIAL(20, 0.3)'
  },
  {
    name: 'ARGO.POISSON',
    description: 'Returns a random sample from a Poisson distribution',
    parameters: ['lambda'],
    example: '=ARGO.POISSON(5)'
  },
  {
    name: 'ARGO.GEOMETRIC',
    description: 'Returns a random sample from a Geometric distribution',
    parameters: ['p'],
    example: '=ARGO.GEOMETRIC(0.3)'
  },
  {
    name: 'ARGO.HYPERGEOMETRIC',
    description: 'Returns a random sample from a Hypergeometric distribution',
    parameters: ['N', 'K', 'n'],
    example: '=ARGO.HYPERGEOMETRIC(50, 20, 10)'
  }
];

console.log('Argo Custom Excel Functions:\n');
customFunctions.forEach((fn, i) => {
  console.log(`${i + 1}. ${fn.name}`);
  console.log(`   ${fn.description}`);
  console.log(`   Parameters: ${fn.parameters.join(', ')}`);
  console.log(`   Example: ${fn.example}\n`);
});

## Part 4: Task Pane UI Components

The add-in includes three main React components:

### 1. Distribution Selector

Visual grid of all 14 distributions with colorful teal icons.

In [ ]:
// DistributionSelector component structure

const distributionSelectorExample = `
export type DistributionType =
  | 'normal' | 'uniform' | 'triangular' | 'lognormal' | 'exponential'
  | 'beta' | 'gamma' | 'weibull' | 'pareto' | 'pert'
  | 'binomial' | 'poisson' | 'geometric' | 'hypergeometric';

interface DistributionInfo {
  id: DistributionType;
  name: string;
  icon: string;               // Path to distribution icon (24x24 or 48x48)
  description: string;
  continuous: boolean;
}

const DISTRIBUTIONS: DistributionInfo[] = [
  {
    id: 'normal',
    name: 'Normal',
    icon: '/assets/distributions/dist-normal-48.png',
    description: 'Bell-shaped, symmetric distribution',
    continuous: true
  },
  // ... 13 more distributions
];

// Component renders a grid of clickable distribution cards
// Selected distribution is highlighted with teal border
`;

console.log('DistributionSelector Component:');
console.log(distributionSelectorExample);

### 2. Simulation Controls

Dynamic parameter forms that adapt to the selected distribution.

In [ ]:
// SimulationControls component structure

const simulationControlsExample = `
// Parameter definitions for each distribution
const DISTRIBUTION_PARAMS: Record<DistributionType, Array<ParamDef>> = {
  normal: [
    { name: 'mean', label: 'Mean (μ)', default: 100 },
    { name: 'stddev', label: 'Std Dev (σ)', default: 15, min: 0.01 }
  ],
  triangular: [
    { name: 'min', label: 'Minimum', default: 50 },
    { name: 'mode', label: 'Most Likely', default: 100 },
    { name: 'max', label: 'Maximum', default: 200 }
  ],
  // ... all 14 distributions
};

// Component features:
// - Dynamic TextField inputs based on selected distribution
// - Iteration count slider (1,000 to 100,000)
// - "Run Simulation" PrimaryButton
// - Spinner progress indicator during execution
`;

console.log('SimulationControls Component:');
console.log(simulationControlsExample);

### 3. Results Dashboard

Visualization of simulation results with Recharts.

In [ ]:
// ResultsDashboard component structure

const resultsDashboardExample = `
export interface SimulationResults {
  samples: number[];    // All generated samples
  mean: number;
  median: number;
  stddev: number;
  min: number;
  max: number;
  p5: number;          // 5th percentile
  p25: number;         // 25th percentile
  p50: number;         // 50th percentile (median)
  p75: number;         // 75th percentile
  p95: number;         // 95th percentile
}

// Dashboard sections:

// 1. Statistics Grid (3 columns)
//    - Mean, Median, Std Dev
//    - Min, Max, Iterations

// 2. Percentiles Bar (horizontal)
//    - P5, P25, P50 (median), P75, P95
//    - Visual dividers between percentiles

// 3. Histogram Chart (Recharts BarChart)
//    - 50 bins across sample range
//    - Teal bars (#01807e)
//    - Tooltip on hover

// 4. CDF Chart (Recharts LineChart)
//    - Cumulative distribution function
//    - Smooth teal curve
//    - 0-100% Y-axis
`;

console.log('ResultsDashboard Component:');
console.log(resultsDashboardExample);

## Part 5: Running Simulations

Here's how the simulation workflow operates in the add-in:

In [ ]:
// Complete simulation workflow

const simulationWorkflow = `
// 1. User selects distribution (e.g., "Normal")
const selectedDistribution = 'normal';

// 2. User enters parameters
const parameters = { mean: 100, stddev: 15 };
const iterations = 10000;

// 3. Create distribution instance from @argo/core
import { NormalDistribution, SimpleRNG } from '@argo/core';

const createDistribution = (type: DistributionType, params: any) => {
  switch (type) {
    case 'normal':
      return new NormalDistribution(params.mean, params.stddev);
    case 'uniform':
      return new UniformDistribution(params.min, params.max);
    // ... all 14 distributions
  }
};

const distribution = createDistribution(selectedDistribution, parameters);
const rng = new SimpleRNG(42);  // Seeded for reproducibility

// 4. Generate samples
const samples: number[] = [];
for (let i = 0; i < iterations; i++) {
  samples.push(distribution.sample(rng));
}

// 5. Calculate statistics
import { calculateStatistics } from '@argo/core';
const results = calculateStatistics(samples);

// 6. Write results to Excel
await Excel.run(async (context) => {
  const sheet = context.workbook.worksheets.getActiveWorksheet();
  
  // Write summary statistics
  const summaryRange = sheet.getRange('A1:B7');
  summaryRange.values = [
    ['Statistic', 'Value'],
    ['Mean', results.mean],
    ['Median', results.median],
    ['Std Dev', results.stddev],
    ['Min', results.min],
    ['Max', results.max],
    ['Iterations', iterations]
  ];
  
  // Format header with Booz Allen teal
  const header = sheet.getRange('A1:B1');
  header.format.font.bold = true;
  header.format.fill.color = '#01807E';
  header.format.font.color = 'white';
  
  await context.sync();
});

// 7. Update dashboard UI
setSimulationResults(results);
`;

console.log('Complete Simulation Workflow:');
console.log(simulationWorkflow);

## Part 6: Design System

The add-in follows the Argo design system with Booz Allen branding.

In [ ]:
// Design system specifications

const designSystem = {
  colors: {
    primary: '#01807E',      // Booz Allen Teal (buttons, headers, icons)
    secondary: '#263846',    // Navy (text, accents)
    background: '#FFFFFF',   // White (main background)
    surface: '#FAF9F8',      // Light gray (cards, panels)
    border: '#E1DFDD',       // Border color
    success: '#107C10',      // Green (success states)
    error: '#D13438',        // Red (error states)
    warning: '#FFB900'       // Yellow (warning states)
  },
  
  typography: {
    fontFamily: 'Segoe UI, -apple-system, BlinkMacSystemFont, sans-serif',
    fontSize: '14px',
    lineHeight: '1.5',
    headerWeight: 600,
    bodyWeight: 400
  },
  
  icons: {
    logo: '/assets/icons/logo-64.png',           // 64x64 app logo
    distributions: '/assets/distributions/',     // 14 distribution icons (24x24, 48x48)
    ribbonCommands: '/assets/icons/',            // Ribbon command icons (32x32, 80x80)
    uiStates: '/assets/ui-states/'               // Loading, error, success, warning (24x24)
  },
  
  layout: {
    taskPaneWidth: '320px',   // Fixed task pane width
    padding: '16px',          // Standard padding
    borderRadius: '8px',      // Card border radius
    spacing: '12px'           // Grid gap
  },
  
  accessibility: {
    contrastRatio: 4.5,       // WCAG 2.1 AA minimum
    focusOutline: '2px solid #01807E',
    ariaLabels: true,         // All interactive elements labeled
    keyboardNavigation: true  // Full keyboard support
  }
};

console.log('Argo Excel Add-in Design System:\n');
console.log(JSON.stringify(designSystem, null, 2));

## Part 7: Manifest.xml Configuration

The manifest.xml defines the add-in's metadata and capabilities.

In [ ]:
// Key manifest.xml sections

const manifestStructure = `
<?xml version="1.0" encoding="UTF-8" standalone="yes"?>
<OfficeApp>
  <!-- Basic metadata -->
  <Id>b5f5f0e0-7c0b-4e4f-9a6e-5d8f0c3b2a1d</Id>
  <Version>5.0.0</Version>
  <ProviderName>Booz Allen Hamilton</ProviderName>
  <DefaultLocale>en-US</DefaultLocale>
  <DisplayName>Argo - Monte Carlo Simulation</DisplayName>
  
  <!-- Permissions -->
  <Permissions>ReadWriteDocument</Permissions>
  
  <!-- Hosts (Excel only) -->
  <Hosts>
    <Host Name="Workbook" />
  </Hosts>
  
  <!-- Task pane UI -->
  <DefaultSettings>
    <SourceLocation DefaultValue="https://localhost:3000/index.html" />
  </DefaultSettings>
  
  <!-- Ribbon buttons -->
  <VersionOverrides>
    <Hosts>
      <Host xsi:type="Workbook">
        <DesktopFormFactor>
          <ExtensionPoint xsi:type="PrimaryCommandSurface">
            <OfficeTab id="TabHome">
              <Group id="ArgoGroup">
                <Label resid="ArgoGroupLabel" />
                <Icon>
                  <bt:Image size="16" resid="Icon.16x16" />
                  <bt:Image size="32" resid="Icon.32x32" />
                  <bt:Image size="80" resid="Icon.80x80" />
                </Icon>
                
                <!-- "Run Simulation" button -->
                <Control xsi:type="Button" id="TaskpaneButton">
                  <Label resid="TaskpaneButtonLabel" />
                  <Icon>
                    <bt:Image size="32" resid="Simulate.32x32" />
                    <bt:Image size="80" resid="Simulate.80x80" />
                  </Icon>
                  <Action xsi:type="ShowTaskpane">
                    <TaskpaneId>ButtonId1</TaskpaneId>
                    <SourceLocation resid="Taskpane.Url" />
                  </Action>
                </Control>
              </Group>
            </OfficeTab>
          </ExtensionPoint>
        </DesktopFormFactor>
      </Host>
    </Hosts>
  </VersionOverrides>
  
  <!-- Custom functions -->
  <ExtensionPoint xsi:type="CustomFunctions">
    <Script>
      <SourceLocation resid="Functions.Script.Url" />
    </Script>
    <Page>
      <SourceLocation resid="Functions.Page.Url" />
    </Page>
    <Metadata>
      <SourceLocation resid="Functions.Metadata.Url" />
    </Metadata>
    <Namespace resid="Functions.Namespace" />
  </ExtensionPoint>
</OfficeApp>
`;

console.log('Manifest.xml Key Sections:');
console.log(manifestStructure);

## Part 8: Building and Deployment

The add-in uses Vite for fast builds and hot module reloading.

In [ ]:
// Build process overview

const buildProcess = {
  development: {
    command: 'npm run dev',
    description: 'Start Vite dev server with HMR',
    url: 'https://localhost:3000',
    features: [
      'Hot Module Replacement (HMR)',
      'Fast refresh for React components',
      'Source maps for debugging',
      'HTTPS with self-signed certificate'
    ]
  },
  
  production: {
    command: 'npm run build',
    description: 'Build optimized production bundle',
    output: 'dist/',
    steps: [
      '1. TypeScript compilation (tsc)',
      '2. Vite build (bundling, minification)',
      '3. Asset optimization',
      '4. Copy manifest.xml to dist/',
      '5. Copy assets/ to dist/assets/'
    ],
    bundleSize: '855 KB (gzipped: 252 KB)'
  },
  
  deployment: {
    appSource: [
      '1. Validate manifest with office-addin-manifest',
      '2. Test sideloading on Windows, Mac, Web',
      '3. Create AppSource submission package',
      '4. Submit to Microsoft Partner Center',
      '5. Wait for certification (2-4 weeks)',
      '6. Publish to AppSource'
    ],
    selfHosted: [
      '1. Build production bundle',
      '2. Deploy dist/ to HTTPS server',
      '3. Update manifest.xml URLs',
      '4. Distribute manifest.xml to users',
      '5. Users sideload via "Upload My Add-in"'
    ]
  }
};

console.log('Build and Deployment Process:\n');
console.log(JSON.stringify(buildProcess, null, 2));

## Part 9: Testing Strategy

The add-in uses Jest with jsdom for React component testing.

In [ ]:
// Testing configuration

const testingStrategy = {
  framework: 'Jest 29 + ts-jest',
  environment: 'jsdom',  // Browser-like environment for React
  
  setup: {
    file: 'tests/setup.ts',
    mocks: [
      'Office.js global object',
      'Excel.run() context',
      'Custom function registration'
    ]
  },
  
  testTypes: [
    {
      type: 'Unit Tests',
      location: 'tests/components/',
      description: 'Test React components in isolation',
      examples: [
        'DistributionSelector renders all 14 distributions',
        'SimulationControls shows correct parameters',
        'ResultsDashboard calculates statistics correctly'
      ]
    },
    {
      type: 'Integration Tests',
      location: 'tests/integration/',
      description: 'Test Office.js interactions',
      examples: [
        'Reading cell ranges works correctly',
        'Writing results to worksheet succeeds',
        'Custom functions register properly'
      ]
    },
    {
      type: 'E2E Tests',
      location: 'Manual testing in Excel',
      description: 'Real Excel testing (Windows required)',
      platform: 'Sprint 12 - Windows with Excel 365'
    }
  ],
  
  coverage: {
    current: '0% (no tests yet - Sprint 11 task)',
    target: '80% for UI components',
    threshold: {
      branches: 0,
      functions: 0,
      lines: 0,
      statements: 0
    },
    passWithNoTests: true  // Currently no tests implemented
  }
};

console.log('Testing Strategy:\n');
console.log(JSON.stringify(testingStrategy, null, 2));

## Part 10: Security and Privacy

The add-in follows Microsoft's security best practices.

In [ ]:
const fs = require('fs');
const path = require('path');

// Read SECURITY.md if it exists
const securityPath = path.join('..', 'packages', 'argo-excel', 'SECURITY.md');
try {
  const securityDoc = fs.readFileSync(securityPath, 'utf-8');
  console.log('Security Documentation (SECURITY.md):\n');
  console.log(securityDoc.substring(0, 1500));
  console.log('\n... (truncated for brevity)');
} catch (err) {
  console.log('SECURITY.md not found or not readable');
  console.log('\nSecurity Principles:');
  console.log('- All processing is LOCAL (no data transmitted)');
  console.log('- Minimal permissions (ReadWriteDocument only)');
  console.log('- No telemetry or analytics collected');
  console.log('- HTTPS required for manifest URLs');
  console.log('- Content Security Policy enforced');
}

## Summary

The Argo Excel Add-in provides a complete Monte Carlo simulation solution for Excel:

### Key Features
- **14 Probability Distributions** - All available as custom functions and in UI
- **Interactive Task Pane** - Visual distribution selector, parameter forms, results dashboard
- **Custom Excel Functions** - Use `ARGO.NORMAL()`, etc. directly in formulas
- **Beautiful Visualizations** - Histogram and CDF charts with Recharts
- **Booz Allen Branding** - Teal/navy color scheme, professional design
- **Cross-Platform** - Works on Windows, Mac, and Web versions of Excel

### Technology Stack
- React 18 + TypeScript 5
- Office.js for Excel integration
- Fluent UI components
- Recharts for data visualization
- Vite for fast builds
- @argo/core simulation engine

### Current Status (Sprint 10)
- ✅ Core UI components complete (DistributionSelector, SimulationControls, ResultsDashboard)
- ✅ All 14 custom functions registered
- ✅ Build system working (855KB bundle)
- ✅ CI/CD passing
- ⏸️ Testing (Sprint 11)
- ⏸️ Accessibility (Sprint 11)
- ⏸️ Real Excel testing (Sprint 12 - requires Windows)

### Next Steps
- **Sprint 11:** Add component tests, accessibility features, advanced UI
- **Sprint 12:** Test in real Excel on Windows, capture screenshots, prepare AppSource submission
- **Future:** Publish to Microsoft AppSource for public availability